In [19]:
import sys, os
sys.path.append('../../official_github/')  # 부모 디렉터리의 파일을 가져올 수 있도록 설정
import pickle
import numpy as np
from collections import OrderedDict
from common.layers import *
from common.gradient import numerical_gradient

In [79]:
import numpy as np
from numba import jit, njit
from numba.typed import Dict
from numba.experimental import jitclass
from numba import int32, float32, float64

In [51]:

np.max(arr, axis=0)

array([4, 5, 6])

In [59]:
@njit
def np_apply_along_axis(func1d, axis, arr):
  assert arr.ndim == 2
  assert axis in [0, 1]
  if axis == 0:
    result = np.empty(arr.shape[1])
    for i in range(len(result)):
      result[i] = func1d(arr[:, i])
  else:
    result = np.empty(arr.shape[0])
    for i in range(len(result)):
      result[i] = func1d(arr[i, :])
  return result

@njit
def np_mean(array, axis):
  return np_apply_along_axis(np.mean, axis, array)

@njit
def np_std(array, axis):
  return np_apply_along_axis(np.std, axis, array)

@njit
def np_sum(array, axis):
  return np_apply_along_axis(np.sum, axis, array)

@njit
def np_max(array, axis):
  return np_apply_along_axis(np.max, axis, array)

In [69]:


np_max(arr, 0)

array([4., 9., 6.])

In [95]:

@njit
def softmax(x):
    x = x.astype('float64')
    if x.ndim == 2:
        x = x.T
        x = x - np_max(x, 0)
        y = np.exp(x) / np.sum(np.exp(x))
        return y.T 

    x = x - np.max(x) # 오버플로 대책
    return np.exp(x) / np.sum(np.exp(x))

def cross_entropy_error(y, t):
    if y.ndim == 1:
        t = t.reshape(1, t.size)
        y = y.reshape(1, y.size)
        
    # 훈련 데이터가 원-핫 벡터라면 정답 레이블의 인덱스로 반환
    if t.size == y.size:
        t = t.argmax(axis=1)
             
    batch_size = y.shape[0]
    return -np.sum(np.log(y[np.arange(batch_size), t])) / batch_size




cross_entropy_error(x, t)

-1.4451858789480823

In [97]:
# spec = [
#     ('y', float64[:]),
#     ('t', int32)
# ]




# @jitclass
# class SoftmaxWithLoss:
#     def __init__(self):
#         # self.loss = None # 손실함수
#         # self.y = None    # softmax의 출력
#         # self.t = None    # 정답 레이블(원-핫 인코딩 형태)
#         pass
        
#     def forward(self, x, t):
#         self.t = t
#         self.y = softmax(x)
#         # self.loss = cross_entropy_error(self.y, self.t)
        
#         # return self.loss
    

#     # def backward(self, dout=1):
#     #     batch_size = self.t.shape[0]
#     #     if self.t.size == self.y.size: # 정답 레이블이 원-핫 인코딩 형태일 때
#     #         dx = (self.y - self.t) / batch_size
#     #     else:
#     #         dx = self.y.copy()
#     #         dx[np.arange(batch_size), self.t] -= 1
#     #         dx = dx / batch_size
        
#     #     return dx
    
# sft = SoftmaxWithLoss()


# x = np.array([[1,2,3],
#                 [4,5,6]])

# t = np.array([[1,2,3],
#                 [4,5,6]])

# sft.forward(x, t)

TypingError: Failed in nopython mode pipeline (step: nopython frontend)
[1m[1m[1m[1m- Resolution failure for literal arguments:
[1mFailed in nopython mode pipeline (step: nopython frontend)
[1m[1mCannot resolve setattr: (instance.jitclass.SoftmaxWithLoss#2062e573990<>).t = array(int32, 2d, C)
[1m
File "..\..\..\..\..\AppData\Local\Temp\ipykernel_5632\2247768986.py", line 18:[0m
[1m<source missing, REPL/exec in use?>[0m
[0m
[0m[1mDuring: typing of set attribute 't' at C:\Users\dieyo\AppData\Local\Temp\ipykernel_5632\2247768986.py (18)[0m
[1m
File "..\..\..\..\..\AppData\Local\Temp\ipykernel_5632\2247768986.py", line 18:[0m
[1m<source missing, REPL/exec in use?>[0m
[0m
[0m[1m- Resolution failure for non-literal arguments:
[1mNone[0m
[0m[0m
[0m[1mDuring: resolving callee type: BoundFunction((<class 'numba.core.types.misc.ClassInstanceType'>, 'forward') for instance.jitclass.SoftmaxWithLoss#2062e573990<>)[0m
[0m[1mDuring: typing of call at <string> (3)
[0m


In [104]:
class SoftmaxWithLoss:
    def __init__(self):
        self.loss = None # 손실함수
        self.y = None    # softmax의 출력
        self.t = None    # 정답 레이블(원-핫 인코딩 형태)
        
    def forward(self, x, t):
        self.t = t
        self.y = softmax(x)
        self.loss = cross_entropy_error(self.y, self.t)
        
        return self.loss

    def backward(self, dout=1):
        batch_size = self.t.shape[0]
        if self.t.size == self.y.size: # 정답 레이블이 원-핫 인코딩 형태일 때
            dx = (self.y - self.t) / batch_size
        else:
            dx = self.y.copy()
            dx[np.arange(batch_size), self.t] -= 1
            dx = dx / batch_size
        
        return dx
    
class Convolution:
    def __init__(self, W, b, stride=1, pad=0):
        self.W = W
        self.b = b
        self.stride = stride
        self.pad = pad
        
        # 중간 데이터（backward 시 사용）
        self.x = None   
        self.col = None
        self.col_W = None
        
        # 가중치와 편향 매개변수의 기울기
        self.dW = None
        self.db = None

    def forward(self, x):
        FN, C, FH, FW = self.W.shape
        N, C, H, W = x.shape
        out_h = 1 + int((H + 2*self.pad - FH) / self.stride)
        out_w = 1 + int((W + 2*self.pad - FW) / self.stride)

        col = im2col(x, FH, FW, self.stride, self.pad)
        col_W = self.W.reshape(FN, -1).T

        out = np.dot(col, col_W) + self.b
        out = out.reshape(N, out_h, out_w, -1).transpose(0, 3, 1, 2)

        self.x = x
        self.col = col
        self.col_W = col_W

        return out

    def backward(self, dout):
        FN, C, FH, FW = self.W.shape
        dout = dout.transpose(0,2,3,1).reshape(-1, FN)

        self.db = np.sum(dout, axis=0)
        self.dW = np.dot(self.col.T, dout)
        self.dW = self.dW.transpose(1, 0).reshape(FN, C, FH, FW)

        dcol = np.dot(dout, self.col_W.T)
        dx = col2im(dcol, self.x.shape, FH, FW, self.stride, self.pad)

        return dx


class Pooling:
    def __init__(self, pool_h, pool_w, stride=1, pad=0):
        self.pool_h = pool_h
        self.pool_w = pool_w
        self.stride = stride
        self.pad = pad
        
        self.x = None
        self.arg_max = None

    def forward(self, x):
        N, C, H, W = x.shape
        out_h = int(1 + (H - self.pool_h) / self.stride)
        out_w = int(1 + (W - self.pool_w) / self.stride)

        col = im2col(x, self.pool_h, self.pool_w, self.stride, self.pad)
        col = col.reshape(-1, self.pool_h*self.pool_w)

        arg_max = np.argmax(col, axis=1)
        out = np.max(col, axis=1)
        out = out.reshape(N, out_h, out_w, C).transpose(0, 3, 1, 2)

        self.x = x
        self.arg_max = arg_max

        return out

    def backward(self, dout):
        dout = dout.transpose(0, 2, 3, 1)
        
        pool_size = self.pool_h * self.pool_w
        dmax = np.zeros((dout.size, pool_size))
        dmax[np.arange(self.arg_max.size), self.arg_max.flatten()] = dout.flatten()
        dmax = dmax.reshape(dout.shape + (pool_size,)) 
        
        dcol = dmax.reshape(dmax.shape[0] * dmax.shape[1] * dmax.shape[2], -1)
        dx = col2im(dcol, self.x.shape, self.pool_h, self.pool_w, self.stride, self.pad)
        
        return dx
    
class Leaky:
    def __init__(self) -> None:
        self.mask = None
    
    def forward(self, x):
        out = np.maximum(0.01 * x, x)
        self.out = out
        return out
            
    def backward(self, dout):
        dx = np.ones_like(self.out)
        dx[self.out < 0] = 0.01
        dx = dout * dx
        
        return dx
    
class Affine:
    def __init__(self, W, b):
        self.W = W
        self.b = b
        
        self.x = None
        self.original_x_shape = None
        # 가중치와 편향 매개변수의 미분
        self.dW = None
        self.db = None

    def forward(self, x):
        # 텐서 대응
        self.original_x_shape = x.shape
        x = x.reshape(x.shape[0], -1)
        self.x = x

        out = np.dot(self.x, self.W) + self.b

        return out

    def backward(self, dout):
        dx = np.dot(dout, self.W.T)
        self.dW = np.dot(self.x.T, dout)
        self.db = np.sum(dout, axis=0)
        
        dx = dx.reshape(*self.original_x_shape)  # 입력 데이터 모양 변경(텐서 대응)
        return dx

In [105]:
@jit
def init(input_dim=(1, 28, 28), 
        conv_param=(30, 5, 0, 1),
        hidden_size=100, output_size=10, weight_init_std=0.01):
    
        filter_num = conv_param[0]
        filter_size = conv_param[1]
        filter_pad = conv_param[2]
        filter_stride = conv_param[3]
        input_size = input_dim[1]
        conv_output_size = (input_size - filter_size + 2*filter_pad) / filter_stride + 1
        pool_output_size = int(filter_num * (conv_output_size/2) * (conv_output_size/2))
        
        
        cnn_weight = weight_init_std * np.random.randn(filter_num, input_dim[0], filter_size, filter_size)
        cnn_bias = np.zeros(filter_num)
        hidden_weight = weight_init_std * np.random.randn(pool_output_size, hidden_size)
        hidden_bias = np.zeros(hidden_size)
        output_weight = weight_init_std * np.random.randn(hidden_size, output_size)
        output_bias = np.zeros(output_size)
        
        ### no numba
        layer_conv1 = Convolution(cnn_weight, cnn_bias, conv_param[3], conv_param[2])
        layer_leaky1 = Leaky()
        
        layer_pool1 = Pooling(pool_h=2, pool_w=2, stride=2)
        
        layer_affine1 = Affine(hidden_weight, hidden_bias)
        layer_leaky2 = Leaky()
        
        layer_affine2 = Affine(output_weight, output_bias)
        
        last_layer = SoftmaxWithLoss()
        ###
        
        
        return cnn_weight, cnn_bias, hidden_weight, hidden_bias, output_weight, output_bias, layer_conv1, layer_leaky1, layer_pool1, layer_affine1, layer_relu2, layer,

init()


C:\Users\dieyo\AppData\Local\Temp\ipykernel_5632\1559630201.py:1: NumbaDeprecationWarning: The 'nopython' keyword argument was not supplied to the 'numba.jit' decorator. The implicit default value for this argument is currently False, but it will be changed to True in Numba 0.59.0. See https://numba.readthedocs.io/en/stable/reference/deprecation.html#deprecation-of-object-mode-fall-back-behaviour-when-using-jit for details.
  @jit
C:\Users\dieyo\AppData\Local\Temp\ipykernel_5632\1559630201.py:1: NumbaWarning: 
Compilation is falling back to object mode WITH looplifting enabled because Function "init" failed type inference due to: Untyped global name 'SoftmaxWithLoss': Cannot determine Numba type of <class 'type'>

File "..\..\..\..\..\AppData\Local\Temp\ipykernel_5632\1559630201.py", line 22:
<source missing, REPL/exec in use?>

  @jit
